# SafeCampus AI — Entrenamiento YOLOv8s (Gun + Knife)

Este notebook entrena un modelo **YOLOv8s** (small) para detectar armas de fuego y cuchillos,
usando un dataset grande de Roboflow (~8,400+ imagenes).

**Cambios vs version anterior:**
- Modelo base: `yolov8s.pt` (small) en vez de `yolov8n.pt` (nano) → menos falsos positivos
- Dataset mas grande: ~8,400 imagenes (gun + knife)
- Guardado automatico en Google Drive (no se pierde si Colab se desconecta)
- Augmentaciones mejoradas para reducir falsos positivos

**Pasos:**
1. Montar Google Drive + verificar GPU
2. Instalar dependencias
3. Descargar dataset de Roboflow
4. Entrenar YOLOv8s con transfer learning
5. Evaluar metricas
6. Probar con imagenes de test
7. Descargar `best.pt`

> **IMPORTANTE:** Antes de ejecutar, ve a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU T4`

In [ ]:
# Paso 1: Montar Google Drive + verificar GPU
from google.colab import drive
drive.mount('/content/drive')

# Carpeta para guardar resultados (persiste si Colab se desconecta)
import os
DRIVE_OUTPUT = "/content/drive/MyDrive/SafeCampus-Training"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Resultados se guardaran en: {DRIVE_OUTPUT}")

# Verificar GPU
!nvidia-smi

In [ ]:
# Paso 2: Instalar dependencias
!pip install ultralytics roboflow -q

In [ ]:
# Paso 3: Descargar dataset de Roboflow
#
# Dataset: "Gun and Knife Detection" por Mahad Ahmed
# Link: https://universe.roboflow.com/mahad-ahmed/gun-and-knife-detection
# ~8,451 imagenes | Clases: gun, knife
#
# Si este dataset no funciona o quieres probar otro, descomenta una alternativa:
#
# ALT 1 — Weapon yolo8 (EDI Detection) ~10,066 imgs
# project = rf.workspace("edi-detection").project("weapon-yolo8")
#
# ALT 2 — Weapon Detection (yolov7test) ~9,672 imgs, muchas clases
# project = rf.workspace("yolov7test-u13vc").project("weapon-detection-m7qso")
#
# ALT 3 — Yolo Weapon Detection ~4,556 imgs (dataset anterior)
# project = rf.workspace("weapon-detect-qbsiw").project("yolo-weapon-detection")

from roboflow import Roboflow

# Obtener tu API key gratis en: https://app.roboflow.com/settings/api
API_KEY = input("Ingresa tu Roboflow API key: ")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("mahad-ahmed").project("gun-and-knife-detection")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset descargado en: {dataset.location}")

In [ ]:
# Paso 3b: Verificar estructura del dataset
import yaml
import os

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

print("=" * 40)
print("DATASET INFO")
print("=" * 40)
print(f"Clases ({config['nc']}): {config['names']}")

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"  {split}: {count} imagenes")
    else:
        print(f"  {split}: no encontrado")

In [ ]:
# Paso 4: Entrenar YOLOv8s
#
# Usamos yolov8s.pt (small) en vez de yolov8n.pt (nano):
#   - nano:  3.2M params, rapido pero mas falsos positivos
#   - small: 11.2M params, mejor precision, aun rapido en GPU
#
# Si quieres probar nano (mas rapido, menos preciso):
#   model = YOLO("yolov8n.pt")

from ultralytics import YOLO

model = YOLO("yolov8s.pt")

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    save=True,
    save_period=10,
    project=OUTPUT_DIR,
    name="gun_knife_v2",
    exist_ok=True,
    # Augmentaciones para reducir falsos positivos
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
)

In [ ]:
# Paso 5: Ver resultados del entrenamiento
from IPython.display import Image, display

RESULTS_DIR = f"{OUTPUT_DIR}/gun_knife_v2"

print("Curvas de entrenamiento:")
display(Image(filename=f"{RESULTS_DIR}/results.png", width=800))

print("\nMatriz de confusion:")
display(Image(filename=f"{RESULTS_DIR}/confusion_matrix.png", width=600))

print("\nPredicciones en validacion:")
display(Image(filename=f"{RESULTS_DIR}/val_batch0_pred.png", width=800))

In [ ]:
# Paso 6: Evaluar en test set
from ultralytics import YOLO

RESULTS_DIR = f"{OUTPUT_DIR}/gun_knife_v2"
best_model = YOLO(f"{RESULTS_DIR}/weights/best.pt")
metrics = best_model.val(data=data_yaml, device=0)

print(f"\n{'=' * 50}")
print(f"  RESULTADOS FINALES — SafeCampus YOLOv8s")
print(f"{'=' * 50}")
print(f"  mAP@50:      {metrics.box.map50:.3f}")
print(f"  mAP@50-95:   {metrics.box.map:.3f}")
print(f"  Precision:   {metrics.box.mp:.3f}")
print(f"  Recall:      {metrics.box.mr:.3f}")
print(f"{'=' * 50}")

# Comparacion con modelo anterior (Threat-Detection-YOLOv8n)
print(f"\n  Modelo anterior (referencia):")
print(f"    mAP@50: 0.813 | Precision: 0.843 | Recall: 0.763")
print(f"\n  Si mAP@50 > 0.813 y Precision > 0.843 → el nuevo modelo es mejor")

In [ ]:
# Paso 7: Probar con imagenes del test set
import glob

test_images = glob.glob(os.path.join(dataset.location, "test", "images", "*"))[:5]
if test_images:
    results = best_model.predict(test_images, conf=0.50, device=0)
    for r in results:
        img = r.plot()
        from PIL import Image as PILImage
        display(PILImage.fromarray(img[:, :, ::-1]))
else:
    print("No hay imagenes de test. Probando con validacion...")
    val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*"))[:5]
    results = best_model.predict(val_images, conf=0.50, device=0)
    for r in results:
        img = r.plot()
        from PIL import Image as PILImage
        display(PILImage.fromarray(img[:, :, ::-1]))

In [ ]:
# Paso 8: Descargar best.pt
from google.colab import files
import shutil

RESULTS_DIR = f"{OUTPUT_DIR}/gun_knife_v2"
src = f"{RESULTS_DIR}/weights/best.pt"
dst = "/content/best_safecampus_v2.pt"

shutil.copy2(src, dst)

# Mostrar tamano del modelo
size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"Modelo: {dst}")
print(f"Tamano: {size_mb:.1f} MB")
print(f"\nTambien guardado en Google Drive: {src}")
print(f"\nCopia este archivo a: safecampus-ai/backend/models/best.pt")

files.download(dst)